# 05 — Synthetic v2, M0: nguyên liệu, thống kê val2019 và chốt catalog cutout

Ngày tạo: 21/10/2026. Nhánh `feat/synthetic-data-v2`. Theo [quyết định D1–D21](../../docs/project/synthetic-data-v2-decisions-2026-10-21.md) và [plan](../../docs/project/synthetic-data-v2-plan-2026-10-20.md), task T1.1–T1.8; theo dõi ở `AN/KE_HOACH_DATA_V2.md`.

**Làm gì:** chạy lại các bước CPU của M0 bằng CLI `tools.synth_v2` và ghi bằng chứng: tập val thiết kế (val2019 bỏ 208 ảnh nguồn nền) và thống kê bố cục; kiểm kê 6 bộ cutout; chốt catalog cutout cuối cùng theo kết quả duyệt bằng mắt (`reviews.json`) và bảng camera theo SKU (`skus.json`).

**Không chạy lại trong notebook:** tải zip Kaggle (27,2 GB, `data/cache/kaggle/`), cắt cutout SAM trên GPU (log ở `runs/data_preprocessing/cutouts_*`), duyệt bằng mắt (trang duyệt ở `runs/data_preprocessing/m0_review/`). Các bước đó đã chạy ngày 21/10 và chỉ được kiểm hash ở đây.

**Input:** `data/archive` (instances, backgrounds, cutouts/sam, train2019 đã giải nén), `data/processed/cutouts_*`, real release v1 (`ann/checkout_native.json`, `task_config.json`). **Output:** `data/processed/rpc_synth_v2_v1/{design,catalog}`, run receipt ở `runs/data_preprocessing/synthetic_v2/<run_id>/`.

In [1]:
import json, os, platform, subprocess, sys, uuid, hashlib, time
from collections import Counter
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pdp/main.py").exists())
os.chdir(REPO); sys.path.insert(0, str(REPO))
TZ = ZoneInfo("Asia/Bangkok")
RUN_ID = uuid.uuid4().hex[:8]
STARTED_AT = datetime.now(TZ).isoformat(timespec="seconds")
RUN_DIR = REPO / "runs/data_preprocessing/synthetic_v2" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
OUT = Path("data/processed/rpc_synth_v2_v1")
CAT = OUT / "catalog"
git = lambda *a: subprocess.run(["git", *a], capture_output=True, text=True).stdout.strip()
ENV = {"run_id": RUN_ID, "started_at": STARTED_AT, "git_revision": git("rev-parse", "HEAD"),
       "git_dirty": bool(git("status", "--porcelain", "--", "autocheckout", "tools", "tests")),
       "python": sys.version.split()[0], "host": platform.node()}
print(json.dumps(ENV, indent=1))

{
 "run_id": "57ab8898",
 "started_at": "2026-10-21T15:34:49+07:00",
 "git_revision": "7b810317d53934adb823300bc69315aeca4868e5",
 "git_dirty": true,
 "python": "3.10.22",
 "host": "archlinux"
}


## 1. Tập val thiết kế và thống kê bố cục (T1.3, D3, D5)

Bỏ 208 ảnh val2019 là nguồn của nền khay; thống kê chỉ đọc val2019 (lệnh từ chối test2019).

In [2]:
def run(*args):
    t = time.time()
    r = subprocess.run([sys.executable, "-m", "tools.synth_v2", *args], capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr[-2000:]); assert r.returncode == 0, r.stderr[-2000:]
    return round(time.time() - t, 1)

print("seconds:", run("stats", "--out-dir", str(OUT / "design"), "--workers", "16"))
stats = json.load(open(OUT / "design/design_statistics.json"))
receipt = json.load(open(OUT / "design/receipt.json"))
print(receipt["counts"])
for level, s in stats["objects_per_image"].items():
    print(level, "objects/image", s["min"], "-", s["max"], "mean", s["mean"], "| bbox covered p50", stats["covered_fraction"][level]["p50"])

design val: 5792 images, 208 excluded -> data/processed/rpc_synth_v2_v1/design
seconds: 4.6
{'val_images': 5792, 'val_objects': 71108, 'excluded': 208, 'levels': {'easy': 1933, 'medium': 1921, 'hard': 1938}}
easy objects/image 4 - 10 mean 7.153 | bbox covered p50 0.3169
medium objects/image 10 - 15 mean 12.377 | bbox covered p50 0.3998
hard objects/image 15 - 20 mean 17.288 | bbox covered p50 0.462


## 2. Kiểm kê 6 bộ cutout (T1.2, T1.5)

`sam_v1` bộ gốc; `cam0_v1` camera 0 cho 200 SKU; `cam2s_v1`, `cam2s_v2`, `cam0s_v2` cắt bù cho SKU ít cutout; `repl_v1` cắt lại bằng SAM 2.1 large các cutout lỗi mask. Mỗi bộ được kiểm: file tồn tại, alpha khác rỗng, ảnh nguồn nằm trong train2019, SHA-256.

In [3]:
SETS = {"sam_v1": "data/archive/cutouts/sam", "cam0_v1": "data/processed/cutouts_cam0_v1",
        "cam2s_v1": "data/processed/cutouts_cam2_supp_v1", "cam2s_v2": "data/processed/cutouts_cam2_supp_v2",
        "cam0s_v2": "data/processed/cutouts_cam0_supp_v2", "repl_v1": "data/processed/cutouts_repair_l_v1"}
for name, d in SETS.items():
    run("inventory", "--index", f"{d}/index.json", "--cutout-set", name, "--root", "data/archive",
        "--out", str(CAT / f"cutouts_{name}.json"))
    issues = json.load(open(CAT / f"cutouts_{name}.json"))["issues"]
    assert not issues, issues[:5]

7502 usable cutouts, 0 issues, cameras {0: 876, 1: 2417, 2: 3143, 3: 1066} -> data/processed/rpc_synth_v2_v1/catalog/cutouts_sam_v1.json


4047 usable cutouts, 0 issues, cameras {0: 4047} -> data/processed/rpc_synth_v2_v1/catalog/cutouts_cam0_v1.json


364 usable cutouts, 0 issues, cameras {2: 364} -> data/processed/rpc_synth_v2_v1/catalog/cutouts_cam2s_v1.json


1430 usable cutouts, 0 issues, cameras {2: 1430} -> data/processed/rpc_synth_v2_v1/catalog/cutouts_cam2s_v2.json


24 usable cutouts, 0 issues, cameras {0: 24} -> data/processed/rpc_synth_v2_v1/catalog/cutouts_cam0s_v2.json


377 usable cutouts, 0 issues, cameras {2: 377} -> data/processed/rpc_synth_v2_v1/catalog/cutouts_repl_v1.json


## 3. Kết quả duyệt và bảng camera theo SKU (T1.6, T1.7)

Duyệt bằng mắt 2 vòng + sửa bằng SAM-L, quyết định của người dùng ngày 21/10 (xem nhật ký trong `AN/KE_HOACH_DATA_V2.md`). Kiểm hash và tóm tắt.

In [4]:
reviews = json.load(open(CAT / "reviews.json"))["reviews"]
skus = json.load(open(CAT / "skus.json"))["skus"]
print("reviews", len(reviews), Counter(r["decision"] for r in reviews))
print("second-review labels", Counter(r["second_review"]["label"] for r in reviews if "second_review" in r))
print("shape groups", Counter(s["shape_group"] for s in skus))
for p in ("reviews.json", "skus.json"):
    print(p, hashlib.sha256((CAT / p).read_bytes()).hexdigest()[:16])

reviews 6625 Counter({'keep': 5903, 'reject': 722})
second-review labels Counter({'disc': 215, 'frag': 185, 'ok': 144, 'twoface': 105, 'broken': 35})
shape groups Counter({'flat': 123, 'upright': 64, 'mixed': 13})
reviews.json 664c761fb8edbc61
skus.json 0323834fae651915


## 4. Chốt M0 (T1.8)

Catalog cuối = cutout `keep` trên camera mà SKU dùng; mỗi annotation nguồn chỉ một cutout. Chặn nếu SKU nào dưới 8 cutout.

In [5]:
run("finalize", "--catalogs", *[str(CAT / f"cutouts_{n}.json") for n in SETS], "--reviews", str(CAT / "reviews.json"),
    "--skus", str(CAT / "skus.json"), "--out-dir", str(CAT), "--accepted-by", "user (chat 21/10/2026)",
    "--notes", "two-face box views kept rejected; turntable fragments rejected; SAM-L repairs accepted per side-by-side review")
final = json.load(open(CAT / "cutouts.json"))["cutouts"]
per_sku = Counter(c["rpc_category_id"] for c in final)
print("final cutouts", len(final), "by set", dict(Counter(c["cutout_set"] for c in final)), "by camera", dict(Counter(c["camera"] for c in final)))
print("per SKU min/median/max", min(per_sku.values()), sorted(per_sku.values())[100], max(per_sku.values()))
print(json.load(open(CAT / "m0_review.json"))["counts"])

M0: 5877 cutouts, min 11/SKU -> data/processed/rpc_synth_v2_v1/catalog
final cutouts 5877 by set {'cam0_v1': 1369, 'cam0s_v2': 2, 'cam2s_v1': 118, 'cam2s_v2': 926, 'repl_v1': 184, 'sam_v1': 3278} by camera {0: 1712, 2: 4165}
per SKU min/median/max 11 20 73
{'cutouts': 5877, 'skus': 200, 'min_per_sku': 11, 'reviews': 6625, 'kept': 5903}


In [6]:
FINISHED_AT = datetime.now(TZ).isoformat(timespec="seconds")
summary = {**ENV, "finished_at": FINISHED_AT, "status": "PASS_M0",
           "outputs": {p: hashlib.sha256((CAT / p).read_bytes()).hexdigest() for p in ("cutouts.json", "m0_review.json", "skus.json", "reviews.json")},
           "limits": ["visual review by one reviewer (Claude) with user confirmation of reject galleries",
                      "SAM GPU extraction and Kaggle download not re-run here",
                      "design statistics use bbox overlap as an occlusion proxy"]}
json.dump(summary, open(RUN_DIR / "summary.json", "w"), indent=1)
print(json.dumps(summary, indent=1))

{
 "run_id": "57ab8898",
 "started_at": "2026-10-21T15:34:49+07:00",
 "git_revision": "7b810317d53934adb823300bc69315aeca4868e5",
 "git_dirty": true,
 "python": "3.10.22",
 "host": "archlinux",
 "finished_at": "2026-10-21T15:36:09+07:00",
 "status": "PASS_M0",
 "outputs": {
  "cutouts.json": "4e975453a7c695f0ae3bc23f26c32baa1252eb7e57b395d4664a5257c588ea9f",
  "m0_review.json": "5d0710138891e0f9d785d18a78dfe3696de6e9f76a88a9822ae2123f0cbebd06",
  "skus.json": "0323834fae651915f055769644a360b9a95004409bd3ab37dafe8c620316ced0",
  "reviews.json": "664c761fb8edbc61a784abe6084b5e407db2a7c2ff0a48968abba70ea2f6ef8a"
 },
 "limits": [
  "visual review by one reviewer (Claude) with user confirmation of reject galleries",
  "SAM GPU extraction and Kaggle download not re-run here",
  "design statistics use bbox overlap as an occlusion proxy"
 ]
}
